In [3]:
!pip install requests beautifulsoup4 pandas

In [1]:
# ============================================
# UFC COMPLETE SCRAPE - FULL VERSION
# Extract fighter_name (first + last), height, reach, stance
# wins, losses, draw, win_rate, total_fights
# ============================================

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

print("="*60)
print("UFC COMPLETE SCRAPE - FULL DATA")
print("="*60)

# ============================================
# STEP 1: GET FIGHTER LIST (FULL NAME + URL)
# ============================================
print("\nSTEP 1: Scraping fighter list...")

base_url = "http://ufcstats.com/statistics/fighters"
fighters_list = []

for page in range(1, 6):
    url = base_url if page == 1 else f"{base_url}?page={page}"
    print(f"  Page {page}...", end=" ")

    try:
        r = requests.get(url, timeout=15)
        soup = BeautifulSoup(r.content, 'html.parser')

        table = soup.find('table', class_='b-statistics__table')
        if not table:
            print("No table found")
            break

        rows = table.find_all('tr')[1:]
        page_count = 0

        for row in rows:
            cols = row.find_all('td')
            if len(cols) >= 3:
                name_cell = cols[0]
                full_name = name_cell.get_text(strip=True)
                fighter_url = name_cell.find('a')['href'] if name_cell.find('a') else None

                height_raw = cols[3].get_text(strip=True) if len(cols) > 3 else None
                weight_raw = cols[4].get_text(strip=True) if len(cols) > 4 else None
                reach_raw = cols[5].get_text(strip=True) if len(cols) > 5 else None
                stance_raw = cols[6].get_text(strip=True) if len(cols) > 6 else None
                wins_raw = cols[7].get_text(strip=True) if len(cols) > 7 else '0'
                losses_raw = cols[8].get_text(strip=True) if len(cols) > 8 else '0'
                draws_raw = cols[9].get_text(strip=True) if len(cols) > 9 else '0'

                fighters_list.append({
                    'fighter_name': full_name,
                    'url': fighter_url,
                    'height_raw': height_raw if height_raw != '--' else None,
                    'weight_raw': weight_raw if weight_raw != '--' else None,
                    'reach_raw': reach_raw if reach_raw != '--' else None,
                    'stance_raw': stance_raw if stance_raw != '--' else None,
                    'wins': wins_raw,
                    'losses': losses_raw,
                    'draws': draws_raw
                })
                page_count += 1

        print(f"Got {page_count} fighters")
        time.sleep(1)

    except Exception as e:
        print(f"Error: {e}")
        break

df_fighters = pd.DataFrame(fighters_list)
print(f"\nTotal fighters from main table: {len(df_fighters)}")

# ============================================
# STEP 2: GET TALE OF THE TAPE DETAILS
# Get height, reach, stance from individual fighter pages
# ============================================
print("\nSTEP 2: Getting Tale of the Tape details...")

def parse_height_to_cm(height_str):
    if not height_str or height_str == '--' or height_str == '0':
        return None
    match = re.search(r"(\d+)'?\s*(\d+)?\"?", height_str)
    if match:
        feet = int(match.group(1))
        inches = int(match.group(2)) if match.group(2) else 0
        return round(feet * 30.48 + inches * 2.54, 1)
    return None

def parse_reach_to_cm(reach_str):
    if not reach_str or reach_str == '--' or reach_str == '0':
        return None
    match = re.search(r'(\d+(?:\.\d+)?)', reach_str)
    if match:
        return round(float(match.group(1)) * 2.54, 1)
    return None

def parse_weight_to_kg(weight_str):
    if not weight_str or weight_str == '--':
        return None
    match = re.search(r'(\d+)', weight_str)
    if match:
        return round(int(match.group(1)) * 0.453592, 1)
    return None

def get_fighter_tott(url):
    try:
        r = requests.get(url, timeout=15)
        soup = BeautifulSoup(r.content, 'html.parser')

        details = {'height': None, 'reach': None, 'stance': None}

        items = soup.select('.b-list__box-list-item')
        for item in items:
            label = item.select_one('.b-list__box-item-title')
            value = item.select_one('.b-list__box-item-value')
            if label and value:
                label_text = label.get_text(strip=True).lower()
                value_text = value.get_text(strip=True)

                if 'height' in label_text:
                    details['height'] = value_text
                elif 'reach' in label_text:
                    details['reach'] = value_text
                elif 'stance' in label_text:
                    details['stance'] = value_text.lower()

        if not details['height'] and not details['reach']:
            tables = soup.find_all('table', class_='b-list__table')
            for table in tables:
                rows = table.find_all('tr')
                for row in rows:
                    th = row.find('th')
                    td = row.find('td')
                    if th and td:
                        label_text = th.get_text(strip=True).lower()
                        value_text = td.get_text(strip=True)

                        if 'height' in label_text:
                            details['height'] = value_text
                        elif 'reach' in label_text:
                            details['reach'] = value_text
                        elif 'stance' in label_text:
                            details['stance'] = value_text.lower()

        return details
    except Exception as e:
        return {'height': None, 'reach': None, 'stance': None}

heights, reaches, stances = [], [], []
total = min(len(df_fighters), 50)

for idx in range(total):
    row = df_fighters.iloc[idx]
    if idx % 10 == 0:
        print(f"  Progress: {idx}/{total}")

    if row['url']:
        tott = get_fighter_tott(row['url'])
        heights.append(tott['height'])
        reaches.append(tott['reach'])
        stances.append(tott['stance'])
    else:
        heights.append(None)
        reaches.append(None)
        stances.append(None)

    time.sleep(0.3)

for idx in range(total):
    df_fighters.loc[idx, 'height_tott'] = heights[idx]
    df_fighters.loc[idx, 'reach_tott'] = reaches[idx]
    df_fighters.loc[idx, 'stance_tott'] = stances[idx]

# ============================================
# STEP 3: NORMALIZE DATA AND CALCULATE METRICS
# ============================================
print("\nSTEP 3: Normalizing data and calculating metrics...")

df_fighters['height_final'] = df_fighters['height_tott'].fillna(df_fighters['height_raw'])
df_fighters['reach_final'] = df_fighters['reach_tott'].fillna(df_fighters['reach_raw'])
df_fighters['stance_final'] = df_fighters['stance_tott'].fillna(df_fighters['stance_raw'])

df_fighters['height_cm'] = df_fighters['height_final'].apply(parse_height_to_cm)
df_fighters['reach_cm'] = df_fighters['reach_final'].apply(parse_reach_to_cm)
df_fighters['weight_kg'] = df_fighters['weight_raw'].apply(parse_weight_to_kg)

df_fighters['stance'] = df_fighters['stance_final'].fillna('unknown')
df_fighters['stance'] = df_fighters['stance'].str.lower()
df_fighters['stance'] = df_fighters['stance'].apply(
    lambda x: 'orthodox' if 'orthodox' in x else
              'southpaw' if 'southpaw' in x else
              'switch' if 'switch' in x else 'unknown'
)

def get_handedness(stance):
    if stance == 'southpaw':
        return 'left'
    elif stance == 'orthodox':
        return 'right'
    elif stance == 'switch':
        return 'ambidextrous'
    return 'unknown'

df_fighters['handedness'] = df_fighters['stance'].apply(get_handedness)

df_fighters['wins'] = pd.to_numeric(df_fighters['wins'], errors='coerce').fillna(0).astype(int)
df_fighters['losses'] = pd.to_numeric(df_fighters['losses'], errors='coerce').fillna(0).astype(int)
df_fighters['draws'] = pd.to_numeric(df_fighters['draws'], errors='coerce').fillna(0).astype(int)

df_fighters['total_fights'] = df_fighters['wins'] + df_fighters['losses'] + df_fighters['draws']
df_fighters['win_rate'] = df_fighters['wins'] / df_fighters['total_fights']
df_fighters['win_rate'] = df_fighters['win_rate'].fillna(0).round(3)

def get_win_group(rate):
    if rate >= 0.9: return '90%+'
    elif rate >= 0.8: return '80-90%'
    elif rate >= 0.7: return '70-80%'
    elif rate >= 0.6: return '60-70%'
    elif rate >= 0.5: return '50-60%'
    elif rate >= 0.4: return '40-50%'
    else: return '<40%'

df_fighters['win_rate_group'] = df_fighters['win_rate'].apply(get_win_group)

# ============================================
# STEP 4: EXPORT TO CSV
# ============================================
print("\nSTEP 4: Exporting to CSV...")

output_columns = [
    'fighter_name', 'stance', 'handedness',
    'height_cm', 'reach_cm', 'weight_kg',
    'wins', 'losses', 'draws', 'total_fights',
    'win_rate', 'win_rate_group'
]

df_output = df_fighters[output_columns].copy()

output_file = 'ufc_complete_data.csv'
df_output.to_csv(output_file, index=False)

print(f"\nDone. File saved: {output_file}")
print(f"   Total fighters: {len(df_output)}")
print(f"   Columns: {df_output.columns.tolist()}")

# ============================================
# STEP 5: DISPLAY STATISTICS
# ============================================
print("\nSTATISTICS:")
print(f"\nStance distribution:")
for stance, count in df_output['stance'].value_counts().items():
    if stance != 'unknown':
        print(f"  {stance}: {count} fighters ({count/len(df_output)*100:.1f}%)")

print(f"\nHandedness distribution:")
for hand, count in df_output['handedness'].value_counts().items():
    if hand != 'unknown':
        print(f"  {hand}: {count} fighters ({count/len(df_output)*100:.1f}%)")

rh_southpaw = df_output[(df_output['handedness'] == 'right') & (df_output['stance'] == 'southpaw')]
print(f"\nRight-handed Southpaw fighters: {len(rh_southpaw)}")

print(f"\nPreview (first 10 rows with known stance):")
df_known = df_output[df_output['stance'] != 'unknown'].head(10)
if len(df_known) > 0:
    print(df_known.to_string())
else:
    print("  No fighters with known stance found in this batch.")

try:
    from google.colab import files
    files.download(output_file)
    print("\nFile downloaded to your computer")
except:
    print(f"\nFile saved as: {output_file}")

UFC COMPLETE SCRAPE - FULL DATA

STEP 1: Scraping fighter list...
  Page 1... Got 25 fighters
  Page 2... Got 24 fighters
  Page 3... Got 24 fighters
  Page 4... Got 24 fighters
  Page 5... Got 24 fighters

Total fighters from main table: 121

STEP 2: Getting Tale of the Tape details...
  Progress: 0/50
  Progress: 10/50
  Progress: 20/50
  Progress: 30/50
  Progress: 40/50

STEP 3: Normalizing data and calculating metrics...

STEP 4: Exporting to CSV...

Done. File saved: ufc_complete_data.csv
   Total fighters: 121
   Columns: ['fighter_name', 'stance', 'handedness', 'height_cm', 'reach_cm', 'weight_kg', 'wins', 'losses', 'draws', 'total_fights', 'win_rate', 'win_rate_group']

STATISTICS:

Stance distribution:
  orthodox: 73 fighters (60.3%)
  southpaw: 19 fighters (15.7%)
  switch: 9 fighters (7.4%)

Handedness distribution:
  right: 73 fighters (60.3%)
  left: 19 fighters (15.7%)
  ambidextrous: 9 fighters (7.4%)

Right-handed Southpaw fighters: 0

Preview (first 10 rows with known

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


File downloaded to your computer
